# 🚢 Supply Chain Agent Workshop
## Building an AI Agent from Scratch with LangChain

### What you will build:
An AI agent that monitors supply chain risks by searching the web in real-time.

### What you will learn:
| Concept | What it means | Why it matters |
|---|---|---|
| **Human-in-the-Loop** | Agent asks a human before acting | Keeps humans in control of high-stakes decisions |
| **Model Flexibility** | Swap GPT ↔ Claude ↔ Gemini easily | You're never stuck with one vendor |
| **Observability** | Every step is logged and traceable | You can debug, audit, and improve your agent |
| **No Vendor Lock-In** | One codebase, any LLM | Business continuity if a provider changes pricing |

---
**Before you start:** Make sure you've copied `.env.example` to `.env` and filled in your API keys.

## Part 1: Setup and Configuration
Run this cell first to install dependencies and verify your config.

In [1]:
# Reload any change .py file before other cells run
%load_ext autoreload
%autoreload 2

# Install dependencies (first run only)

import sys, os
sys.path.insert(0, '..')

import langchain
# LangChain 0.3.x has a global verbose flag that's separate from the AgentExecutor one. 
# Set them to False
langchain.verbose = False
langchain.debug = False

from dotenv import load_dotenv
load_dotenv('../.env')

# Set environment variables for LangSmith tracing
os.environ['LANGCHAIN_TRACING_V2'] = os.getenv('LANGCHAIN_TRACING_V2', 'false')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY', '')
os.environ['LANGCHAIN_PROJECT'] = 'supply-chain-workshop'

print('✅ Environment loaded')
print(f'   Provider: {os.getenv("LLM_PROVIDER", "openai")}')
print(f'   LangSmith: {os.getenv("LANGCHAIN_TRACING_V2", "false")}')

✅ Environment loaded
   Provider: openai
   LangSmith: true


## Part 2: Understanding the Tools
Tools are how the agent interacts with the outside world.
Each tool is a Python function + a description the LLM reads.

> **Key Insight:** The LLM decides WHICH tool to use based on the description.
> Writing good tool descriptions is a critical skill.

In [2]:
from langchain_community.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults

# Initialize Tavily search — built specifically for AI agents
# Sign up free at https://tavily.com to get your API key
tavily = TavilySearchResults(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    search_depth="advanced",
    include_answer=True,
    include_raw_content=False,
    days=30,  # Only return results from the last 30 days
)

# Test the search tool directly (before connecting it to the agent)
result = tavily.run("Port of Vancouver shipping delays March 2026")
print("🔍 Direct Search Result:")
print(str(result)[:500])


🔍 Direct Search Result:
[{'url': 'https://www.everstream.ai/risk-centers/port-congestion-report-12-march-2026/', 'content': 'The Port of Houston, United States, is facing average wait times of 1.8 days, according to Kuehne + Nagel, with Hapag-Lloyd reporting that one crane is currently out of service at the port.  \n\nThe Port of Vancouver, Canada, is facing average wait times of 1.6 days, according to Kuehne + Nagel, with high average rail dwell times and high yard density. \n\nIn South America, average wait times wer


In [3]:
# Wrap Tavily as a LangChain Tool
# The description is what the LLM reads to decide when to use this tool

supply_chain_search = Tool(
    name="supply_chain_search",
    func=tavily.run,
    description=(
        "Search for current supply chain news, disruptions, port delays, "
        "supplier financial issues, trade route problems, or tariff changes. "
        "Input should be a specific search query."
    ),
)

# WORKSHOP EXERCISE: Add a second tool for checking freight rates
# Hint: use tavily.run with a query that includes freight rate or shipping cost
# YOUR CODE HERE:
# freight_rate_tool = Tool(
#     name="...",
#     func=...,
#     description="...",
# )

tools = [supply_chain_search]
print(f"✅ {len(tools)} tool(s) ready")
for t in tools:
    print(f"   - {t.name}: {t.description[:80]}...")


✅ 1 tool(s) ready
   - supply_chain_search: Search for current supply chain news, disruptions, port delays, supplier financi...


## Part 3: Observability — Watching the Agent Think
Before we build the agent, let's build the logging system.

> **Key Insight:** Without observability, an agent is a black box.
> With it, you can see every thought, every tool call, every result.
> This is essential for debugging and for building client trust.

In [4]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.schema import AgentAction, AgentFinish, LLMResult

class WorkshopObservabilityHandler(BaseCallbackHandler):
    
    def __init__(self):
        self.iteration = 0

    def on_llm_start(self, serialized, prompts, **kwargs):
        print('\n🧠 Agent is thinking...', flush=True)

    def on_chat_model_start(self, serialized, messages, **kwargs):
        print('\n🧠 Agent is thinking...', flush=True)
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        self.iteration += 1
        print(f'\n🔄 [ITERATION {self.iteration}]', flush=True)
        print(f'🔧 Using tool: {serialized["name"]}', flush=True)
        print(f'   Query: {input_str[:100]}', flush=True)
    
    def on_tool_end(self, output, **kwargs):
        print(f'📄 Got result ({len(output)} chars)', flush=True)
    
    def on_agent_finish(self, finish: AgentFinish, **kwargs):
        print(f'\n✅ Agent finished!', flush=True)

    # WORKSHOP EXERCISE: Add a method to log token usage
    # Hint: override on_llm_end and inspect response.llm_output

handler = WorkshopObservabilityHandler()
print('✅ Observability handler ready')

✅ Observability handler ready


## Part 4: Model Flexibility
This is the section that prevents vendor lock-in.
We'll build a factory function that returns any LLM.

> **Key Insight:** The agent code never references OpenAI or Anthropic directly.
> It only works with a generic `BaseChatModel`. This is the abstraction layer.

In [5]:
def get_llm(provider='openai', model_name=None, temperature=0.0):
    """
    Workshop demo: swap providers by changing ONE argument.
    The agent below never needs to change.
    """
    if provider == 'openai':
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model_name or 'gpt-4o-mini', temperature=temperature)
    
    elif provider == 'anthropic':
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model_name or 'claude-3-haiku-20240307', temperature=temperature)
    
    elif provider == 'google':
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model_name or 'gemini-1.5-flash', temperature=temperature)
    
    raise ValueError(f'Unknown provider: {provider}')

# Test that it works (will fail if API key is missing — that's expected)
try:
    llm = get_llm(provider='openai')
    print(f'✅ LLM ready: {llm}')
except Exception as e:
    print(f'⚠️  LLM init error (check your API key): {e}')

✅ LLM ready: client=<openai.resources.chat.completions.completions.Completions object at 0x114127440> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1142a3740> root_client=<openai.OpenAI object at 0x11381c170> root_async_client=<openai.AsyncOpenAI object at 0x1139adeb0> model_name='gpt-4o-mini' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********')


## Part 5: Building the Agent
Now we put it all together.

The **ReAct pattern** (Reason + Act) is the most common agent pattern:
1. **Thought**: What do I need to do?
2. **Action**: Call a tool
3. **Observation**: Read the tool result
4. Repeat until ready to answer

In [10]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from datetime import datetime

# ‼️ Critical for accuracy of search results
today = datetime.now().strftime("%B %Y")  # e.g. "April 2026"

PROMPT = PromptTemplate.from_template(f"""
You are a Supply Chain Risk Analyst. Today's date is {today}.

CRITICAL RULES — FOLLOW WITHOUT EXCEPTION:
1. You MUST call a tool at least once before giving a Final Answer
2. Never answer from memory — always search first
3. Always include the current month and year in your search queries
4. If a search returns irrelevant results, try ONE different query then move on
5. NEVER repeat the same search query twice — if it failed once, it will fail again
6. After 3 tool calls, you MUST write your Final Answer using whatever you found

Tools available:
{{tools}}

Available tool names: {{tool_names}}

You MUST use EXACTLY this format:

Thought: I need to search for current information about this.
Action: supply_chain_search
Action Input: [your search query including {today}]
Observation: [tool result — filled in automatically]
Thought: I now have enough information for a final answer.
Final Answer: SEVERITY: [LOW|MEDIUM|HIGH]
SUMMARY: [2-3 sentences based on search results]
RECOMMENDATION: [one specific action]

Question: {{input}}
{{agent_scratchpad}}
""")

# Note the double curly braces {{tools}} — because we're using an f-string, single braces get 
# consumed by Python, so LangChain's template variables need to be doubled.

def build_agent(provider='openai'):
    llm = get_llm(provider=provider)
    agent = create_react_agent(llm=llm, tools=tools, prompt=PROMPT)
    executor = AgentExecutor(
        agent=agent,
        tools=tools,
        callbacks=[handler],
        verbose=False,
        max_iterations=4,
        handle_parsing_errors=True,
        return_intermediate_steps=True,  # ← captures every step reliably
    )
    return executor

agent = build_agent(provider='openai')  # Change provider here!
print('✅ Agent assembled and ready')

✅ Agent assembled and ready


## Part 6: Human-in-the-Loop
The agent found some results — now a human decides what to do with them.

> **Key Insight:** Automation doesn't mean no humans.
> HITL means the RIGHT decisions go to humans, routine ones don't.

In [11]:
def human_approval_gate(risk_summary, risk_level='MEDIUM'):
    """Pause and ask for human approval before alerting the team."""
    print('\n' + '='*55)
    print('🔔 ALERT REQUIRES YOUR APPROVAL')
    print('='*55)
    print(f'Risk Level: {risk_level}')
    print(f'\nSummary:\n{risk_summary[:500]}')
    print('\n' + '-'*55)
    
    response = input('\nSend alert to operations team? (yes/no): ').strip().lower()
    if response in ('yes', 'y'):
        print('✅ Alert approved!')
        return True
    else:
        print('🚫 Alert discarded.')
        return False

print('✅ Human-in-the-Loop gate ready')
print()
print('WORKSHOP EXERCISE:')
print('  Modify human_approval_gate() to:')
print('  1. Only ask for approval on HIGH risk (skip MEDIUM/LOW)')
print('  2. Log all decisions to a file with a timestamp')
print('  3. Add a third option: "escalate" that Cc\'s a manager')

✅ Human-in-the-Loop gate ready

WORKSHOP EXERCISE:
  Modify human_approval_gate() to:
  1. Only ask for approval on HIGH risk (skip MEDIUM/LOW)
  2. Log all decisions to a file with a timestamp
  3. Add a third option: "escalate" that Cc's a manager


## Part 7: Run the Full Agent
This cell runs the complete pipeline:
Search → Reason → Assess Risk → Human Approval → Alert

In [12]:
# WORKSHOP: Try different queries and observe how the agent reasons
queries = [
    'Canada port disruptions shipping delays April 2026',
    'Vancouver port shipping news 2026',
    'North America supply chain disruptions this week',
]

query = queries[1]
print(f'Running query: {query}\n')

result = agent.invoke({'input': query})
output = result.get('output', '')
steps = result.get('intermediate_steps', [])

# Show iterations
print(f"\n🔄 Agent used {len(steps)} tool call(s):\n")
for i, (action, observation) in enumerate(steps, 1):
    print(f"[ITERATION {i}]")
    print(f"🔧 Tool: {action.tool}")
    print(f"   Input: {action.tool_input}")
    print(f"   Result: {str(observation)[:200]}\n")

print('=' * 55)
print('AGENT ASSESSMENT:')
print(output)

# ── HITL Gate ──────────────────────────────────────────
ENABLE_HITL = True   # ← set False during testing, True for demo

if ENABLE_HITL and 'HIGH' in output.upper():
    human_approval_gate(risk_summary=output, risk_level='HIGH')
else:
    print('\n📊 Risk logged. HITL skipped (set ENABLE_HITL=True to activate).')

Running query: Vancouver port shipping news 2026


✅ Agent finished!

🔄 Agent used 1 tool call(s):

[ITERATION 1]
🔧 Tool: supply_chain_search
   Input: Vancouver port shipping news April 2026
   Result: [{'url': 'https://www.globenewswire.com/news-release/2026/01/06/3213788/0/en/DP-World-on-Track-for-Mid-2026-Launch-of-Short-Sea-Shipping-Facility-in-Vancouver.html', 'content': "Accessibility: Skip To

AGENT ASSESSMENT:
SEVERITY: MEDIUM  
SUMMARY: The Vancouver port is experiencing significant developments in April 2026, including the upcoming launch of a CAD$22 million short-sea shipping facility aimed at improving coastal freight services and reducing truck traffic. Additionally, the port is set to welcome a record number of cruise ship visits, with nearly 360 expected this season, indicating a strong recovery in tourism and maritime activities.  
RECOMMENDATION: Monitor the progress of the short-sea shipping facility and assess its impact on logistics and supply chain efficiency in t

## Part 8: Model Swap Challenge
The ultimate test — change the provider and re-run without touching anything else.

In [9]:
# WORKSHOP CHALLENGE:
# 1. Run this cell with provider='openai' — note the response style
# 2. Change to provider='anthropic' — run again
# 3. Change to provider='google' — run again
# 4. Discuss: What was different? What was the same?

PROVIDER = 'openai'  # ← CHANGE THIS: 'openai' | 'anthropic' | 'google'

agent_v2 = build_agent(provider=PROVIDER)
result = agent_v2.invoke({
    'input': 'Search for any disruptions affecting automotive parts supply chains in North America.'
})
print(result.get('output', ''))


✅ Agent finished!
SEVERITY: HIGH
SUMMARY: The automotive parts supply chain in North America is currently facing significant disruptions due to a combination of geopolitical tensions, tariff uncertainties, and ongoing logistical challenges exacerbated by the Strait of Hormuz crisis. These factors have led to production halts, increased operational costs, and a reevaluation of supply chain strategies among manufacturers. The industry is grappling with a structural reality of persistent disruptions rather than cyclical challenges.
RECOMMENDATION: Automotive companies should develop and implement robust contingency plans that include diversifying suppliers and exploring alternative shipping routes to mitigate the impact of ongoing disruptions.


## Workshop Summary

You built an agent that demonstrates four enterprise-grade patterns:

| Pattern | Where in the code | Real-world benefit |
|---|---|---|
| **Human-in-the-Loop** | `human_approval_gate()` | Operations team reviews before alerts go out |
| **Model Flexibility** | `get_llm(provider=...)` | Switch vendors in seconds, not weeks |
| **No Vendor Lock-In** | Agent uses `BaseChatModel` interface | Works with any future LLM without rewriting |
| **Observability** | `WorkshopObservabilityHandler` | Full audit trail of every decision |

### Next Steps
1. Add a tool that reads from your internal supplier database
2. Connect the approval gate to Slack instead of console input
3. Enable LangSmith tracing and explore the visual dashboard
4. Add memory so the agent remembers past risk assessments